# Lab 7 · IA sobre grafos para detectar anomalías de red

Notebook para aplicar detección de anomalías no supervisada con **Isolation Forest** y z-scores contextuales por tipo de nodo. El dataset `node_features.csv` incluye variables estructurales y comportamiento operativo.

In [ ]:
# Instalación de dependencias básicas. En SageMaker normalmente ya están instaladas.
%pip -q install pandas numpy matplotlib networkx scikit-learn s3fs

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Por defecto se leen los CSV incluidos en la carpeta local data/.
# Si prefieres leer desde S3, cambia USE_S3=True y ajusta S3_PREFIX.
USE_S3 = False
DATA_DIR = Path('data')
S3_PREFIX = 's3://TU_BUCKET/TU_CARPETA'  # ejemplo: s3://mi-bucket/graph

def data_path(filename):
    return f"{S3_PREFIX}/{filename}" if USE_S3 else str(DATA_DIR / filename)

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

import networkx as nx
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, confusion_matrix

## Parte A. Cargar y explorar features

In [ ]:
feat = pd.read_csv(data_path('node_features.csv'))
print(feat.shape)
display(feat.head())
display(feat.groupby('node_type')[['traffic_gbps','utilization','error_count_24h','avg_latency_ms','reroute_events_24h']].mean().round(2))

## Parte B. Detección global con Isolation Forest

In [ ]:
vars_comportamiento = ['traffic_gbps','utilization','error_count_24h','avg_latency_ms','reroute_events_24h']
X = StandardScaler().fit_transform(feat[vars_comportamiento])
iso = IsolationForest(contamination=0.10, random_state=42)
feat['anomaly_score_raw'] = iso.fit_predict(X)  # -1 anomalía
feat['anomaly_iso'] = (feat['anomaly_score_raw'] == -1).astype(int)

display(feat[feat['anomaly_iso'] == 1][['node_id','node_type','region','utilization','error_count_24h','avg_latency_ms','reroute_events_24h']])

## Parte C. Detección contextual con z-scores por tipo

In [ ]:
def zscore_por_tipo(df, col):
    media = df.groupby('node_type')[col].transform('mean')
    std = df.groupby('node_type')[col].transform('std').replace(0, 1)
    return (df[col] - media) / std

for c in ['utilization','error_count_24h','avg_latency_ms','reroute_events_24h']:
    feat[f'z_{c}'] = zscore_por_tipo(feat, c)

feat['z_max'] = feat[['z_utilization','z_error_count_24h','z_avg_latency_ms','z_reroute_events_24h']].max(axis=1)
feat['anomaly_z'] = (feat['z_max'] > 2.5).astype(int)
display(feat[feat['anomaly_z'] == 1][['node_id','node_type','region','z_max']].sort_values('z_max', ascending=False))

## Parte D. Validar contra anomalías inyectadas

In [ ]:
for metodo in ['anomaly_iso','anomaly_z']:
    p = precision_score(feat['is_anomaly'], feat[metodo], zero_division=0)
    r = recall_score(feat['is_anomaly'], feat[metodo], zero_division=0)
    print(f'{metodo}: precision={p:.2f}, recall={r:.2f}')
    print(confusion_matrix(feat['is_anomaly'], feat[metodo]))

feat['anomaly_combo_or'] = ((feat['anomaly_iso']==1) | (feat['anomaly_z']==1)).astype(int)
feat['anomaly_combo_and'] = ((feat['anomaly_iso']==1) & (feat['anomaly_z']==1)).astype(int)
for metodo in ['anomaly_combo_or','anomaly_combo_and']:
    p = precision_score(feat['is_anomaly'], feat[metodo], zero_division=0)
    r = recall_score(feat['is_anomaly'], feat[metodo], zero_division=0)
    print(f'{metodo}: precision={p:.2f}, recall={r:.2f}')

In [ ]:
cols = ['node_id','node_type','region','is_anomaly','anomaly_iso','anomaly_z','anomaly_combo_or','anomaly_combo_and','z_max'] + vars_comportamiento
feat[cols].sort_values(['anomaly_combo_or','z_max'], ascending=False).to_csv(OUTPUT_DIR/'lab07_resultados_anomalias.csv', index=False)
display(feat[cols].sort_values(['anomaly_combo_or','z_max'], ascending=False).head(12))

## Parte E. Visualizar anomalías sobre el grafo

In [ ]:
nodes = pd.read_csv(data_path('network_nodes.csv'))
edges = pd.read_csv(data_path('network_edges.csv'))
G = nx.Graph()
for _, r in nodes.iterrows():
    G.add_node(r['node_id'], node_type=r['node_type'])
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'])

anom = set(feat[feat['anomaly_combo_or'] == 1]['node_id'])
colors = ['tab:red' if n in anom else 'tab:blue' for n in G.nodes()]
labels = {n:n for n in anom}
pos = nx.spring_layout(G, seed=42, k=0.5)
plt.figure(figsize=(13,9))
nx.draw_networkx_edges(G, pos, alpha=0.30)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=260)
nx.draw_networkx_labels(G, pos, labels, font_size=8)
plt.title('Nodos detectados como anómalos')
plt.axis('off'); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab07_anomalias_grafo.png', dpi=150, bbox_inches='tight')
plt.show()

## Cierre operativo

Una anomalía estadística no es automáticamente un incidente. Para el entregable, clasifica cada nodo detectado como posible incidente real o falso positivo y propón un flujo de triaje humano.